In [ ]:
import os
import math
import numpy as np
import xarray as xr
import scipy.ndimage
from pathlib import Path
import netCDF4 as nc
import cc3d
import gc
import metpy
from metpy.units import units
import metpy.calc as mpcalc
import matplotlib.pyplot as plt
import scipy.linalg

#Constants
L_v = 2.5 * (10**6) #latent heat
c_pd = 1004 #specific heat
p_0 = 10**5 #reference pressure
R_d = 287
R_v = 461.5

t = 6
t_conds = 18

In [ ]:
#Open datasets
ds_ql = xr.open_dataset("/mnt/stor-pool-01/projects/heus/EUREC4A_Eulerian/Feb_1st_12day_cdnc70_nudge/ql.nc", decode_times=False,chunks={'time': 1}) #liquid water mixing ratio
ds_qt = xr.open_dataset("/mnt/stor-pool-01/projects/heus/EUREC4A_Eulerian/Feb_1st_12day_cdnc70_nudge/qt.nc", decode_times=False,chunks={'time': 1}) #water mixing ratio
ds_th_l = xr.open_dataset("/mnt/stor-pool-01/projects/heus/EUREC4A_Eulerian/Feb_1st_12day_cdnc70_nudge/thl.nc", decode_times=False,chunks={'time': 1}) #liquid virtual potential temperature
ds_p = xr.open_dataset("/mnt/stor-pool-01/projects/heus/EUREC4A_Eulerian/Feb_1st_12day_cdnc70_nudge/p.nc", decode_times=False,chunks={'time': 1}) #pressure
ds_initial_thermo_conds = xr.open_dataset("/mnt/stor-pool-01/projects/heus/EUREC4A_Eulerian/Feb_1st_12day_cdnc70_nudge/eurec4a.default.0000000.nc",group='thermo', decode_times=False,chunks={'time': 1}).isel(time=t_conds) #initial conds
ds_initial_general_conds = xr.open_dataset("/mnt/stor-pool-01/projects/heus/EUREC4A_Eulerian/Feb_1st_12day_cdnc70_nudge/eurec4a.default.0000000.nc",group='default', decode_times=False,chunks={'time': 1}).isel(time=t_conds) #initial conds

In [ ]:
#Calculate B
chi = R_d / c_pd
big_pi = (ds_p.p / p_0) ** chi
ds_thv = (ds_th_l.thl + (L_v / (c_pd * big_pi)) * ds_ql.ql) * (1 - (1 - (R_v / R_d)) - ((R_v / R_d) * ds_qt.qt))
thv_mean = ds_thv.mean(dim=['y', 'x'], skipna=True)
da_b = (9.81/300) * (ds_thv - thv_mean)
ds_b = xr.DataArray.to_dataset(da_b,name="b")

In [ ]:
#Calculate VPG
ds_vpg = ds_p.p.differentiate("z")

In [ ]:
#Function
def solve_generalized_poisson_3d(f, sigma, z_coords, dx, dy):
    """
    Solves the 3D generalized Poisson equation on a vertically stretched grid:
        grad dot (sigma(z) * grad(m)) = f(x, y, z)
    """
    nz, ny, nx = f.shape
    
    # 1. Horizontal Fourier Transform of the RHS source term
    f_hat = np.fft.rfft2(f, axes=(1, 2))
    
    # 2. Compute symmetric horizontal wavenumbers squared (kx^2 + ky^2)
    kx = np.fft.rfftfreq(nx, d=dx) * 2 * np.pi
    ky = np.fft.fftfreq(ny, d=dy) * 2 * np.pi
    k_sq = ky[:, np.newaxis]**2 + kx[np.newaxis, :]**2
    nkx = len(kx)
    
    # 3. Compute exact stretched grid metrics from true simulation centers
    dz_center = np.diff(z_coords)                 # Distance between centers (nz - 1)
    zh = 0.5 * (z_coords[:-1] + z_coords[1:])     # Reconstructed half-levels (nz - 1)
    dz_cell = np.diff(zh)                         # Thickness of interior cells (nz - 2)
    
    dz_cell_bottom = z_coords[1] - z_coords[0]
    dz_cell_top = z_coords[-1] - z_coords[-2]
    
    sigma_half = 0.5 * (sigma[:-1] + sigma[1:])   # Length: nz - 1
    
    # 4. Pre-compute the shared vertical operators (Built ONCE outside loops)
    interior_lower = sigma_half[:-1] / (dz_cell * dz_center[:-1])
    interior_upper = sigma_half[1:] / (dz_cell * dz_center[1:])
    
    # Allocate template for the base tridiagonal matrix
    ab_base = np.zeros((3, nz), dtype=complex)
    
    # Fill vertical terms for interior rows
    ab_base[2, :-2] = interior_lower               # Corrected lower diagonal shift
    ab_base[0, 2:]  = interior_upper               # Corrected upper diagonal shift
    ab_base[1, 1:-1] = -interior_upper - interior_lower
    
    # Fill vertical terms for boundary cap rows (Neumann boundary conditions)
    ab_base[1, 0]  = -sigma_half[0] / (dz_cell_bottom * dz_center[0])
    ab_base[0, 1]  =  sigma_half[0] / (dz_cell_bottom * dz_center[0])
    
    ab_base[1, -1] = -sigma_half[-1] / (dz_cell_top * dz_center[-1])
    ab_base[2, -2] =  sigma_half[-1] / (dz_cell_top * dz_center[-1])
    
    # Initialize container for the Fourier-space solution
    m_hat = np.zeros_like(f_hat, dtype=complex)
    
    # 5. Fast Execution Loop
    for r in range(ny):
        for c in range(nkx):
            ksq_rc = k_sq[r, c]
            
            if r == 0 and c == 0:
                m_hat[:, r, c] = 0.0
                continue
                
            # Copy the pre-computed base vertical structure 
            ab = ab_base.copy()
            
            # Layer the horizontal curvature decay on top of the main diagonal
            ab[1, :] -= sigma * ksq_rc
            
            # Instantaneous 1D tridiagonal solver execution
            m_hat[:, r, c] = scipy.linalg.solve_banded((1, 1), ab, f_hat[:, r, c])
            
    # 6. Transform back to real physical space
    m = np.fft.irfft2(m_hat, s=(ny, nx), axes=(1, 2))
    
    return m

In [ ]:
#Obtaining sigma (part of the left hand side)
th = mpcalc.potential_temperature(ds_initial_general_conds.p * units.pascal, ds_initial_thermo_conds.T * units.kelvin) / units.kelvin
#thl_profile = ds_initial_thermo_conds.thl
sigma_numpy = (c_pd * ds_initial_thermo_conds.th).compute().values
if sigma_numpy.ndim > 1: #leakage protection
    sigma_numpy = sigma_numpy[0]

In [ ]:
#Obtaining f' (right hand side)
B_slice = ds_b.b.isel(time=t)
rho_profile = ds_initial_thermo_conds.rhoref

f_field = rho_profile * B_slice
df_dz = f_field.differentiate(coord="z")

f_numpy = df_dz.compute().values
f_numpy = np.nan_to_num(f_numpy, nan=0.0) #clean f

In [ ]:
#dx, dy, z vals
dx = float(ds_ql.x[1] - ds_ql.x[0])
dy = float(ds_ql.y[1] - ds_ql.y[0])
z_numpy = ds_ql.z.compute().values

In [ ]:
#Performing Fourier analysis to find pi'_b
pi_b_numpy = solve_generalized_poisson_3d(
    f=f_numpy, 
    sigma=sigma_numpy, 
    z_coords=z_numpy, 
    dx=dx, 
    dy=dy
)

pi_b_xr = xr.DataArray(
    pi_b_numpy, 
    coords={"z": ds_ql.z, "y": ds_ql.y, "x": ds_ql.x}, 
    dims=["z", "y", "x"]
)

In [ ]:
# =====================================================================
# VERIFICATION CHECKER FOR PI_B (Identical Stencil Matching)
# =====================================================================
print("\n--- Running Spectral & Staggered Verification ---")

# 1. Reconstruct Solver Horizontal Wavenumbers
nx, ny = pi_b_xr.sizes["x"], pi_b_xr.sizes["y"]
kx = np.fft.rfftfreq(nx, d=dx) * 2 * np.pi
ky = np.fft.fftfreq(ny, d=dy) * 2 * np.pi
k_sq = ky[:, np.newaxis]**2 + kx[np.newaxis, :]**2

# 2. Compute Horizontal Laplacian Perfectly in Spectral Space
pi_b_hat = np.fft.rfft2(pi_b_numpy, axes=(1, 2))
sigma_3d = sigma_numpy[:, np.newaxis, np.newaxis]
horizontal_laplacian_hat = -sigma_3d * k_sq[np.newaxis, :, :] * pi_b_hat
horizontal_laplacian_spectral = np.fft.irfft2(horizontal_laplacian_hat, s=(ny, nx), axes=(1, 2))

# 3. Compute Vertical Divergence using Solver Staggered Grid Metrics
dz_center = np.diff(z_numpy)
zh = 0.5 * (z_numpy[:-1] + z_numpy[1:])
dz_cell = np.diff(zh)
sigma_half = 0.5 * (sigma_numpy[:-1] + sigma_numpy[1:])

# Vertical Flux at half-levels (j + 1/2) -> Shape: (nz-1, ny, nx)
flux_z_half = sigma_half[:, np.newaxis, np.newaxis] * (np.diff(pi_b_xr.values, axis=0) / dz_center[:, np.newaxis, np.newaxis])

# Flux Divergence at interior cell centers (j) -> Shape: (nz-2, ny, nx)
diff_2_z_interior = np.diff(flux_z_half, axis=0) / dz_cell[:, np.newaxis, np.newaxis]

# 4. Combine LHS Components & Extract Truth Target (Interior Rows 1 to nz-1)
test_lhs_spectral = horizontal_laplacian_spectral[1:-1, :, :] + diff_2_z_interior
truth_interior = f_numpy[1:-1, :, :]


# 5. Numerical Execution Verification
correlation = np.corrcoef(test_lhs_spectral.flatten(), truth_interior.flatten())[0, 1]
print(f"Pure Spectral-Aligned Spatial Correlation: {correlation:.6f}")

if np.allclose(test_lhs_spectral, truth_interior, rtol=1e-2, atol=1e-2):
    print("Verification: SUCCESS - Solver matches mathematical formulation exactly!")
else:
    print("Verification: FAILURE - Discrepancies found.")

In [ ]:
z_interior = ds_ql.z.isel(z=slice(1, -1))

test_LHS = xr.DataArray(
    test_lhs_spectral,
    coords={
        "z": z_interior,
        "y": ds_ql.y,
        "x": ds_ql.x
    },
    dims=["z", "y", "x"],
    name="spectral_relative_error"
)

test_RHS = xr.DataArray(
    truth_interior,
    coords={
        "z": z_interior,
        "y": ds_ql.y,
        "x": ds_ql.x
    },
    dims=["z", "y", "x"],
    name="spectral_relative_error"
)

In [ ]:
test_min = -0.0009765625
test_max = 0.0009765625

In [ ]:
test_LHS.where(test_LHS != 0).sel(z=1000, method="nearest").plot(x="x",y="y",vmin=test_min,vmax=test_max,cmap='RdBu_r')

In [ ]:
test_RHS.where(test_RHS != 0).sel(z=1000, method="nearest").plot(x="x",y="y",vmin=test_min,vmax=test_max,cmap='RdBu_r')

In [ ]:
#VPG B
#Equation: - cp * theta p, 0 * d(pi_b)/dz
dpi_dz_xr = pi_b_xr.differentiate(coord="z")
vpg_b = -c_pd * th * dpi_dz_xr


In [ ]:
ds_b.b.isel(time=t).sel(z=1000, method="nearest").plot(x="x",y="y")

In [ ]:
ds_vpg.isel(time=0).sel(z=1000, method="nearest").plot(x="x",y="y")

In [ ]:
vpg_b_max = 0.0625#float(vpg_b.max())
vpg_b_min = -0.0625#float(vpg_b.min())
vpg_b.where(vpg_b != 0).sel(z=1000, method="nearest").plot(x="x",y="y",vmin=vpg_b_min,vmax=vpg_b_max,cmap='RdBu_r')

In [ ]:
#--Plotting error in worst case scenario--
# 1. Compute first horizontal derivatives
dpi_dx_xr = pi_b_xr.differentiate(coord="x")
dpi_dy_xr = pi_b_xr.differentiate(coord="y")

# 2. Align sigma as an Xarray DataArray for broadcasting
sigma_da = xr.DataArray(sigma_numpy, coords={"z": pi_b_xr.z}, dims=["z"])

# 3. Calculate horizontal and vertical components of the fluxes
flux_x = sigma_da * dpi_dx_xr
flux_y = sigma_da * dpi_dy_xr
flux_z = sigma_da * pi_b_xr.differentiate(coord="z")

# 4. Take the divergence of those fluxes to find the total estimated LHS
diff_2_x = flux_x.differentiate(coord="x")
diff_2_y = flux_y.differentiate(coord="y")
diff_2_z = flux_z.differentiate(coord="z")

# Resulting LHS
test_lhs_xarray = (diff_2_x + diff_2_y + diff_2_z)

da_error = (test_lhs_xarray - df_dz) / df_dz * 100


In [ ]:
err_min = 0.015625#float(da_error.min())
err_max = -0.015625#float(da_error.max())
da_error.where(da_error != 0).sel(z=1000, method="nearest").plot(x="x",y="y",vmin=-100,vmax=100,cmap='RdBu_r')

In [ ]:
# 1. Calculate the raw relative error array (Shape: nz-2, ny, nx)
err_fft_numpy = (test_lhs_spectral - truth_interior) / truth_interior

# 2. Slice the original z-coordinate to match the interior rows (indices 1 to nz-1)
z_interior = ds_ql.z.isel(z=slice(1, -1))

# 3. Wrap the NumPy array into an Xarray DataArray with aligned coordinates
da_error_fft = xr.DataArray(
    err_fft_numpy,
    coords={
        "z": z_interior,
        "y": ds_ql.y,
        "x": ds_ql.x
    },
    dims=["z", "y", "x"],
    name="spectral_relative_error"
)

# Multiply by 100 to scale it into true percentage space (-100% to 100%)
da_error_fft = da_error_fft * 100.0

In [ ]:
# 1. Isolate the 2D spatial slice at your chosen height
error_2d = da_error.sel(z=1000, method="nearest")
err_fft_2d = da_error_fft.sel(z=1000, method="nearest")

# 2. Flatten and drop any NaN values completely
error_flat = error_2d.values.flatten()
err_ff_flat = err_fft_2d.values.flatten()
error_clean = error_flat[~np.isnan(error_flat)]
err_fft_clean = err_ff_flat[~np.isnan(error_flat)]

# 3. Create the percentage distribution plot
plt.figure(figsize=(8, 5))

# Calculate weights based on the cleaned array length
weights = np.ones_like(error_clean) * 100.0 / len(error_clean)
weights_fft = np.ones_like(err_fft_clean) * 100.0 / len(err_fft_clean)

# Explicitly set the range from -100% to 100%
plt.hist(error_clean, bins=150, range=(-150, 150), weights=weights,
         color="black", edgecolor="black", alpha=0.5)
plt.hist(err_fft_clean, bins=150, range=(-150, 150), weights=weights_fft,
         color="red", edgecolor="red", alpha=0.5)

plt.title("Error Percentage Distribution at z=1000m")
plt.xlabel("Numerical Error Percentage (%)")
plt.ylabel("Percentage of Grid Points (%)")

# Snap the x-axis precisely to your data boundaries
plt.xlim(-150, 150)
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
pi_b_xr.where(pi_b_xr != 0).sel(z=1000, method="nearest").plot(x="x",y="y",vmin=err_min,vmax=err_max,cmap='RdBu_r')

In [ ]:
#Not used functions
""""
def diff_l2_norm (p, p_ref):
    #note both p and p_ref should be a numpy.ndarray
    l2_diff = (np.sqrt(np.sum((p - p_ref)**2)) / np.sqrt(np.sum(p_ref**2)))
    return l2_diff

def laplace_2d (p0, max_iter=10000, rtol=1e-6):
    #--Parameters--
    #p0 : input (numpy.ndarray)
    #max_iter : max iterations
    #rtol : relative tolerance

    #--Output--
    #p : solution
    #ite : number of iterations
    #diff : final relative L2 norm of the difference

    p = p0.copy()
    diff = rtol + 1.0 #initial diff
    ite = 0 #index
    while diff > rtol and ite < max_iter:
        pn = p.copy()
        #Update the solution at interior points
        p[1:-1, 1:-1] = 0.25 * (p[1:-1, :-2] + p[1:-1, 2:] + p[:-2, 1:-1] + p[2:, 1:-1])
        #Apply Neumann condition (zero gradient)
        #-right boundary
        p[1:-1, -1] = p[1:-1, -2]
        #Compute the residual as the L2 norm of the difference
        diff = diff_l2_norm(p, pn)
        ite += 1
    return p, ite, diff
"""